#  World Happiness Report – Interaktive Analyse
### Datenquelle: [Kaggle – World Happiness Report](https://www.kaggle.com/datasets/unsdsn/world-happiness)



In [18]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
df = pd.read_csv("2015_cleaned.csv")


## Plot 1 – Ranking: Die glücklichsten & unglücklichsten Länder 

In [19]:
top10 = df.nlargest(10, "Happiness Score").sort_values("Happiness Score")
bottom10 = df.nsmallest(10, "Happiness Score").sort_values("Happiness Score")

combined = pd.concat([bottom10, top10])
combined["Kategorie"] = ["Bottom 10"] * 10 + ["Top 10"] * 10

fig = go.Figure()

fig.add_trace(go.Bar(
    x=combined["Happiness Score"],
    y=combined["Country"],
    orientation="h",
    marker_color=combined["Kategorie"].map({"Top 10": "#2ecc71", "Bottom 10": "#e74c3c"}),
    text=combined["Happiness Score"].round(2),
    textposition="outside"
))

fig.update_layout(
    title_text="Top 10 vs. Bottom 10 – Happiness Score 2015",
    xaxis_title="Happiness Score",
    yaxis_title="Land",
    height=650,
    template="plotly_white",
    showlegend=False,
    xaxis=dict(range=[0, 9])
)

fig.show()

In [21]:
df[["Country", "Happiness Score", "Dystopia Residual"]].sort_values("Dystopia Residual", ascending=False).head(30)

,Country,Happiness Score,Dystopia Residual
13,Mexico,7.187,3.60214
15,Brazil,6.983,3.26001
22,Venezuela,6.810,3.19131
11,Costa Rica,7.226,3.17728
51,Moldova,5.889,3.10712
80,Pakistan,5.194,3.10709
10,Israel,7.278,3.08854
93,Mozambique,4.971,3.05137
41,El Salvador,6.130,3.03500
77,Nigeria,5.268,2.89319


## Plot 2 – Macht Geld glücklich? BIP pro Kopf vs. Happiness Score

In [ ]:
korr_cols = ["Happiness Score", "Economy (GDP per Capita)", "Family",
             "Health (Life Expectancy)", "Freedom", "Generosity", "Trust (Government Corruption)"]

korr_matrix = df[korr_cols].corr().round(2)

fig = px.imshow(
    korr_matrix,
    text_auto=True,
    color_continuous_scale="RdBu",
    zmin=-1, zmax=1,
    title="🔥 Korrelationsmatrix – Zusammenhänge zwischen allen Faktoren",
    template="plotly_white",
    aspect="auto"
)

fig.update_layout(height=500)
fig.show()
print("💡 Tipp: Werte nahe +1 = starker positiver Zusammenhang, nahe -1 = negativer Zusammenhang")


💡 Tipp: Werte nahe +1 = starker positiver Zusammenhang, nahe -1 = negativer Zusammenhang


## Plot 3 – Korrelations-Heatmap: Was hängt womit zusammen?

In [17]:
fig = px.scatter(
    df,
    x="Economy (GDP per Capita)",
    y="Happiness Score",
    text="Country",
    color="Happiness Score",
    color_continuous_scale="RdYlGn",
    hover_name="Country",
    hover_data={"Economy (GDP per Capita)": True, "Happiness Rank": True},
    title="BIP pro Kopf vs. Happiness Score",
    labels={"Economy (GDP per Capita)": "BIP pro Kopf", "Happiness Score": "Happiness Score"},
    template="plotly_white",
    trendline="ols"
)

fig.update_traces(textposition="top center", textfont_size=9, marker_opacity=0.8)
fig.update_layout(height=600, coloraxis_showscale=True)
fig.show()

## 🗺️ Plot 4 – Weltkarte: Glück rund um den Globus

In [ ]:
fig = px.choropleth(
    df,
    locations="Country",
    locationmode="country names",
    color="Happiness Score",
    hover_name="Country",
    hover_data={
        "Happiness Score": True,
        "Happiness Rank": True,
        "Economy (GDP per Capita)": True
    },
    color_continuous_scale="RdYlGn",
    title="🗺️ World Happiness Score 2015 – Weltkarte",
    labels={"Happiness Score": "Happiness Score"}
)

fig.update_layout(
    height=550,
    template="plotly_white",
    geo=dict(showframe=False, showcoastlines=True)
)
fig.show()


C:\Users\silas\AppData\Local\Temp\ipykernel_2284\661091255.py:1: DeprecationWarning: The library used by the *country names* `locationmode` option is changing in an upcoming version. Country names in existing plots may not work in the new version. To ensure consistent behavior, consider setting `locationmode` to *ISO-3*.
  fig = px.choropleth(


## 🕸️ Plot 5 – Radar-Chart: Top 5 Länder im Vergleich

In [ ]:
top5 = df.nlargest(5, "Happiness Score")
kategorien = ["Economy (GDP per Capita)", "Family",
              "Health (Life Expectancy)", "Freedom", "Generosity", "Trust (Government Corruption)"]
fig = go.Figure()

farben = px.colors.qualitative.Set2
for i, (_, row) in enumerate(top5.iterrows()):
    werte = [row[k] for k in kategorien]
    fig.add_trace(go.Scatterpolar(
        r=werte + [werte[0]],
        theta=kategorien + [kategorien[0]],
        fill="toself",
        name=row["Country"],
        line_color=farben[i],
        opacity=0.7
    ))

fig.update_layout(
    polar=dict(radialaxis=dict(visible=True, range=[0, 1.6])),
    title="🕸️ Radar-Chart – Top 5 Länder im Faktoren-Vergleich",
    height=550,
    template="plotly_white"
)
fig.show()


In [ ]:
import plotly.graph_objects as go
import numpy as np

# Schwellenwerte definieren
gdp_schwelle = df["Economy (GDP per Capita)"].quantile(0.4)   # untere 40% = "arm"
family_schwelle = df["Family"].quantile(0.6)                   # obere 40% = "hoher Family-Wert"

# Länder kategorisieren
def kategorisieren(row):
    if row["Economy (GDP per Capita)"] < gdp_schwelle and row["Family"] > family_schwelle:
        return "Arm aber starke Gemeinschaft"
    elif row["Family"] < df["Family"].quantile(0.3):
        return "Schwacher Family-Wert"
    else:
        return "Andere"

df["Kategorie"] = df.apply(kategorisieren, axis=1)

farben = {
    "Arm aber starke Gemeinschaft": "#1D9E75",
    "Schwacher Family-Wert":        "#D85A30",
    "Andere":                       "#B4B2A9"
}

fig = go.Figure()

for kategorie, farbe in farben.items():
    teil = df[df["Kategorie"] == kategorie]
    fig.add_trace(go.Scatter(
        x=teil["Economy (GDP per Capita)"],
        y=teil["Happiness Score"],
        mode="markers+text",
        name=kategorie,
        text=teil["Country"],
        textposition="top center",
        textfont=dict(size=8),
        marker=dict(
            size=teil["Family"] * 20,   # Punktgröße = Family-Wert
            color=farbe,
            opacity=0.85,
            line=dict(width=1, color="white")
        ),
        hovertemplate=(
            "<b>%{text}</b><br>"
            "BIP: %{x:.2f}<br>"
            "Happiness: %{y:.2f}<br>"
            "Family: %{customdata:.2f}<extra></extra>"
        ),
        customdata=teil["Family"]
    ))

fig.update_layout(
    title="Einfluss von Familie & Gemeinschaft auf das Glück",
    xaxis_title="BIP pro Kopf",
    yaxis_title="Happiness Score",
    template="plotly_white",
    height=600,
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="left", x=0)
)

fig.show()